<a href="https://colab.research.google.com/github/vaishnavi123-cOde/OpenSSL-LibOQS-C-Demonstration-Classical-Post-Quantum-Cryptography-/blob/main/OpenSSL_%26_LibOQS_C_Demonstration_(Classical_%26_Post_Quantum_Cryptography).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install OpenSSL & Build LibOQS from source
!apt-get update -qq
!apt-get install -y libssl-dev cmake build-essential

!git clone -b main https://github.com/open-quantum-safe/liboqs.git
%cd liboqs
!mkdir build
%cd build
!cmake -DCMAKE_INSTALL_PREFIX=/usr -DBUILD_SHARED_LIBS=ON ..
!make -j$(nproc)
!sudo make install
!ldconfig
%cd /content

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libssl-dev is already the newest version (3.0.13-0ubuntu3.15).
cmake is already the newest version (3.28.3-1build7).
build-essential is already the newest version (12.10ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 71 not upgraded.
Cloning into 'liboqs'...
remote: Enumerating objects: 71225, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 71225 (delta 279), reused 238 (delta 238), pack-reused 70897 (from 4)
Receiving o

In [ ]:
%%writefile main.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <openssl/evp.h>
#include <openssl/rand.h>
#include <oqs/oqs.h>

// Helper function to print hex representation of byte arrays
void print_hex(const char *label, const uint8_t *data, size_t len) {
    printf("%s (size: %zu bytes):\n  ", label, len);
    for (size_t i = 0; i < len; i++) {
        printf("%02x", data[i]);
        if ((i + 1) % 32 == 0 && i + 1 < len) printf("\n  ");
    }
    printf("\n\n");
}

// TASK 1: OpenSSL Classical AES-256-GCM Encryption
int demo_openssl_aes_gcm(void) {
    printf("=== [TASK 1] OpenSSL AES-256-GCM Encryption ===\n\n");

    const uint8_t plaintext[] = "Confidential Payload for Cryptographic Library Verification";
    int plaintext_len = (int)strlen((const char *)plaintext);

    uint8_t key[32]; // AES-256
    uint8_t iv[12];  // GCM standard 96-bit IV
    uint8_t ciphertext[128];
    uint8_t tag[16]; // 128-bit authentication tag
    int len = 0, ciphertext_len = 0;

    // Generate secure random key and IV
    RAND_bytes(key, sizeof(key));
    RAND_bytes(iv, sizeof(iv));

    EVP_CIPHER_CTX *ctx = EVP_CIPHER_CTX_new();
    if (!ctx) return -1;

    // Initialize Encryption
    if (1 != EVP_EncryptInit_ex(ctx, EVP_aes_256_gcm(), NULL, key, iv)) return -1;

    // Process Plaintext
    if (1 != EVP_EncryptUpdate(ctx, ciphertext, &len, plaintext, plaintext_len)) return -1;
    ciphertext_len = len;

    // Finalize Encryption
    if (1 != EVP_EncryptFinal_ex(ctx, ciphertext + len, &len)) return -1;
    ciphertext_len += len;

    // Extract GCM Authentication Tag
    if (1 != EVP_CIPHER_CTX_ctrl(ctx, EVP_CTRL_GCM_GET_TAG, 16, tag)) return -1;

    EVP_CIPHER_CTX_free(ctx);

    print_hex("Plaintext Input", plaintext, plaintext_len);
    print_hex("Generated Symmetric Key (256-bit)", key, sizeof(key));
    print_hex("Initialization Vector (IV)", iv, sizeof(iv));
    print_hex("Ciphertext Output", ciphertext, ciphertext_len);
    print_hex("GCM Auth Tag", tag, sizeof(tag));

    return 0;
}

// TASK 2: LibOQS Post-Quantum Kyber-768 Key Encapsulation (KEM)
int demo_liboqs_kyber(void) {
    printf("=== [TASK 2] LibOQS Kyber-768 (PQC KEM) Key Exchange ===\n\n");

    // Initialize KEM instance for Kyber-768
    OQS_KEM *kem = OQS_KEM_new(OQS_KEM_alg_kyber_768);
    if (kem == NULL) {
        printf("Error: Kyber-768 algorithm not supported by liboqs build.\n");
        return -1;
    }

    uint8_t *public_key = malloc(kem->length_public_key);
    uint8_t *secret_key = malloc(kem->length_secret_key);
    uint8_t *ciphertext = malloc(kem->length_ciphertext);
    uint8_t *shared_secret_enc = malloc(kem->length_shared_secret);
    uint8_t *shared_secret_dec = malloc(kem->length_shared_secret);

    if (!public_key || !secret_key || !ciphertext || !shared_secret_enc || !shared_secret_dec) {
        printf("Memory allocation failed.\n");
        return -1;
    }

    // 1. Generate Kyber-768 Keypair
    if (OQS_KEM_keypair(kem, public_key, secret_key) != OQS_SUCCESS) return -1;
    print_hex("Kyber-768 Public Key (Truncated)", public_key, 64);

    // 2. Encapsulate (Client derives shared secret & ciphertext)
    if (OQS_KEM_encaps(kem, ciphertext, shared_secret_enc, public_key) != OQS_SUCCESS) return -1;
    print_hex("Kyber-768 Ciphertext (Truncated)", ciphertext, 64);
    print_hex("Encapsulated Shared Secret (Client)", shared_secret_enc, kem->length_shared_secret);

    // 3. Decapsulate (Server recovers shared secret using secret key)
    if (OQS_KEM_decaps(kem, shared_secret_dec, ciphertext, secret_key) != OQS_SUCCESS) return -1;
    print_hex("Decapsulated Shared Secret (Server)", shared_secret_dec, kem->length_shared_secret);

    // 4. Verify shared secrets match
    if (memcmp(shared_secret_enc, shared_secret_dec, kem->length_shared_secret) == 0) {
        printf("[SUCCESS] Shared secrets match perfectly! PQC KEM exchange verified.\n\n");
    } else {
        printf("[ERROR] Shared secret mismatch!\n\n");
    }

    // Clean up memory
    OQS_MEM_cleanse(secret_key, kem->length_secret_key);
    OQS_MEM_cleanse(shared_secret_enc, kem->length_shared_secret);
    OQS_MEM_cleanse(shared_secret_dec, kem->length_shared_secret);

    free(public_key);
    free(secret_key);
    free(ciphertext);
    free(shared_secret_enc);
    free(shared_secret_dec);
    OQS_KEM_free(kem);

    return 0;
}

int main(void) {
    printf("====================================================\n");
    printf("  CRYPTO LAB: OpenSSL & LibOQS C Demonstration\n");
    printf("====================================================\n\n");

    if (demo_openssl_aes_gcm() != 0) {
        printf("OpenSSL AES-GCM execution failed.\n");
    }

    if (demo_liboqs_kyber() != 0) {
        printf("LibOQS Kyber-768 execution failed.\n");
    }

    return 0;
}

Writing main.c


In [ ]:
# Cell 3: Compile & Run
!gcc main.c -o crypto_demo -lcrypto -loqs
!./crypto_demo


  CRYPTO LAB: OpenSSL & LibOQS C Demonstration

=== [TASK 1] OpenSSL AES-256-GCM Encryption ===

Plaintext Input (size: 59 bytes):
  436f6e666964656e7469616c205061796c6f616420666f722043727970746f67
  726170686963204c69627261727920566572696669636174696f6e

Generated Symmetric Key (256-bit) (size: 32 bytes):
  6a69e244258705749945c4b2ae0148ed894ab22296972592942b22c9ceb00d4b

Initialization Vector (IV) (size: 12 bytes):
  6abd06fd9f212dd607fe188d

Ciphertext Output (size: 59 bytes):
  1ed4dc47afa63bebfa4f793a2c8a86501bface8182bf0bfefd4c080dcb546e39
  91ebbf38ba010b9c6dc6cb2d30a2c54a0eb8ce37fbfc91ce85657e

GCM Auth Tag (size: 16 bytes):
  48fe962fbfd117e951277427a15b60d4

=== [TASK 2] LibOQS Kyber-768 (PQC KEM) Key Exchange ===

Kyber-768 Public Key (Truncated) (size: 64 bytes):
  70d66e56411786a986a72bb8760b6f9130a7e6500484511d4d33b2408c28aada
  8b73e89bed7433cc969256129502a201d08276403828abd026af589e2cf87bef

Kyber-768 Ciphertext (Truncated) (size: 64 bytes):
  9a5581a7c7600174205d48be0e